In [38]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
import os
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import matplotlib.colors as mcolors
import matplotlib.markers as mmarkers
import seaborn as sns
import scipy

from compcats.localizer import localizer
from compcats.datasets.fmri import nsd
from compcats.utils import model_utils

import main_analysis as ma
import paths

# Plot settings
import matplotlib as mpl
from matplotlib import rcParams
rcParams.update({'figure.autolayout': False})
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Avenir'

RESULTS_DIR = 'results'
MODELS_DIR = 'models'
PLOTS_DIR = f'{RESULTS_DIR}/plots/preprint'
os.makedirs(PLOTS_DIR, exist_ok=True)

NSD_DIR = paths.NSD_DIR
GRILLSPECTOR_LOCALIZER_DIR = paths.GRILLSPECTOR_LOCALIZER_DIR
KONKLE_LOCALIZER_DIR = paths.KONKLE_LOCALIZER_DIR
KANWISHER_LOCALIZER_DIR = paths.KANWISHER_LOCALIZER_DIR


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Utils

def get_shades(hex_color, n=5, lighten_limit=0.25, darken_limit=0.75):
    """
    Generate n shades around a base hex_color.
    Base sits in the middle, lighter tints above, darker shades below.
    
    lighten_limit: how much lighter the lightest color can get (0-1)
    darken_limit: how much darker the darkest color can get (0-1)
    """
    rgb = np.array(mcolors.to_rgb(hex_color))
    colors = []
    mid = (n - 1) / 2
    
    for i in range(n):
        factor = (i - mid) / mid  # ranges -1..1
        if factor < 0:
            # lighter: move towards white but limited
            new_rgb = rgb + (1 - rgb) * (-factor * lighten_limit)
        else:
            # darker: move towards black but limited
            new_rgb = rgb * (1 - factor * darken_limit)
        colors.append(np.clip(new_rgb, 0, 1))
    return colors

# Main figure (TODO)

# Supplementary figures

## Fig S5: best model across t-thresholds

## Fig S6: IoU

## Fig S7: Voxel-tuning matched

In [43]:
savefig = True

plot_dir = f'{PLOTS_DIR}/section4/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

tthreshold = 7
frois = ['FFA', 'PPA', 'EBA']
model_localizers = ['grillspector', 'nsd']

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

all_task_optimized_model_names = ma.get_model_names(model_type='task_optimized')

subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
modelsubject_correlations_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

# frois = subjectsubject_correlations_csv['fmri_froi'].unique()
subjects = subjectsubject_correlations_csv['subject1'].unique()
models = modelsubject_correlations_csv['model_name'].unique()

taskoptimized_models = [model for model in models if model in all_task_optimized_model_names]

hh_color = '#fde0c5' # Light peach
# random_color = '#f5e6d3' # Very light brown

localizer_color_dict = {
    'grillspector': '#ffffff',  # Pure white for Grill-Spector localizer
    'nsd': '#ce9393'            # Light red for NSD localizer
}
localizer_name_dict = {
    'grillspector': 'Category-contrast\n(L1, Stigliani et al., 2015)',
    'kanwisher': 'Kanwisher',
    'nsd': 'Voxel-matched'
}

metrics = ['normed_pearsonr', 'normed_rdmr']
lims_dict = {
    'normed_pearsonr': (0, 1.2),
    'normed_rdmr': (0, 1.4)
}
ticks_dict = {
    'normed_pearsonr': [0, 0.5, 1],
    'normed_rdmr': [0, 0.5, 1]
}

# fig, ax = plt.figure(figsize=(5, 5), dpi=300)
fig, axes = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

for midx, metric in enumerate(metrics):
    print()
    print(metric)
    for fidx, froi in enumerate(frois):
        ax = axes[midx, fidx]
        print(froi)
        hh_medians = []
        for subject in subjects:
            subjectsubject_froi = subjectsubject_correlations_csv[
                (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                (subjectsubject_correlations_csv['subject1'] == subject) &
                (subjectsubject_correlations_csv['fmri_tthreshold'] == tthreshold)
            ]
            hh_medians += list(subjectsubject_froi[metric])
        hh_medians = np.array(hh_medians).flatten()

        # Model correlations
        model_corrs = {}
        for model_localizer in model_localizers:
            model_corrs[model_localizer] = {}

            # Separate task-optimized and brain-optimized models
            to_medians = []
            for model in taskoptimized_models:
                modelsubject_froi = modelsubject_correlations_csv[
                    (modelsubject_correlations_csv['fmri_froi'] == froi) &
                    (modelsubject_correlations_csv['model_name'] == model) &
                    (modelsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_localizer'] == model_localizer)
                ]
                model_median = modelsubject_froi[metric].median()
                # Mann-Whitney U test for each model vs. hh_medians
                # ustat, pval = scipy.stats.mannwhitneyu(model_medians, hh_medians, alternative='less')
                # if pval >= 0.05:
                #     print(f"Model {model} ({model_localizer}) is plausibly within the human population (p={pval:.2e})")
                to_medians.append(model_median)
            model_corrs[model_localizer]['task_optimized'] = np.array(to_medians)

        jitter_scale = 0.005 #0.01
        scatter_pad = 0 # 0.05
        alpha = 0.75 # 0.5
        markersize = 15 # 25

        bar_positions = np.array([0, 0.35, 0.7])

        # y=0 line
        ax.axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=0.75, zorder=0)

        # Box plot for brain-brain, and brain-ann for each model localizer
        box_data = [hh_medians]
        for model_localizer in model_localizers:
            box_data.append(model_corrs[model_localizer]['task_optimized'])
        linewidth = 0.75
        barwidth = 0.075 # 0.35
        medianlineextra = 0.075
        boxes = ax.boxplot(
                    box_data, positions=bar_positions, patch_artist=True, 
                    widths=barwidth, showfliers=False,
                    boxprops={'color': 'black', 'linewidth': linewidth},
                    whiskerprops=dict(linewidth=linewidth),
                    capprops=dict(linewidth=linewidth),
                    medianprops={'color': 'black', 'linewidth': linewidth},
                    flierprops=dict(marker='o', markersize=5, linewidth=linewidth)
                )
        for i, data in enumerate(box_data):
            med = np.median(data)
            # Manually draw a longer median line
            x = bar_positions[i]  # X-position of the boxplot
            # 0.225
            ax.plot([x - medianlineextra, x + medianlineextra], [med, med], color='black', linewidth=linewidth, solid_capstyle='butt')
        
        # Color the boxes according to category colors
        colors = [hh_color] + [localizer_color_dict[model_localizer] for model_localizer in model_localizers]
        for box, color in zip(boxes['boxes'], colors):
            box.set(facecolor=color)
        
        # Scatter individual data points
        for i, data in enumerate(box_data):
            x_ = np.zeros(len(data)) + bar_positions[i] + np.random.normal(0, jitter_scale, len(data))
            x_ += scatter_pad
            ax.scatter(
                x_, data, color='gray', alpha=alpha, zorder=5,
                edgecolors='none', s=markersize, linewidths=0.15
            )
            # print(f'Bar position {bar_positions[i]}: n={len(data)}')
        
        # Write froi name in bottom left corner
        ax.text(0.05, 0.25, froi, transform=ax.transAxes, fontsize=12, verticalalignment='top')

        yticks = np.array(ticks_dict[metric])
        ylims = lims_dict[metric]
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_position(('outward', 5))
        ax.spines['bottom'].set_position(('outward', 5))
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticks, fontsize=12)
        ax.spines['left'].set_bounds(yticks.min(), yticks.max())
        # plt.tick_params(axis='y', labelleft=False)
        ax.set_ylim(ylims)
        ax.spines['bottom'].set_bounds(bar_positions[0], bar_positions[-1])
        ax.set_xlim(bar_positions[0]-0.15, bar_positions[-1]+0.15)
        # ax.tick_params(axis='x', labelbottom=False)

        if fidx != 0:
            ax.spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            ax.yaxis.set_ticks([])
        else:
            if metric == 'normed_pearsonr':
                ax.set_ylabel(r'Noise-corrected pearson $\rho$', fontsize=14)
            else:
                ax.set_ylabel(r'Noise-corrected RDM $\rho$', fontsize=14)
        # x-label on all plots for now
        # ax.set_xticks(bar_positions, ['Brain-Brain', 'Brain-ANN\n(Task-Optimized)', 'Brain-ANN\n(Brain-Optimized)'], fontsize=12)
        xtick_labels = ['Brain-Brain'] + \
            [f'Brain-ANN\n{localizer_name_dict[model_localizer]}' for model_localizer in model_localizers]
        ax.set_xticks(bar_positions)
        ax.set_xticklabels(xtick_labels, fontsize=10, rotation=0, ha='center')

plt.tight_layout()
if savefig:
    # raise NotImplementedError("Don't overwrite existing fig3_sup_otherfrois.pdf without checking it first!")
    plt.savefig(f'{plot_dir}/figS7_voxeltuningmatched.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
# plt.savefig(f'{plot_dir}/modelhuman_univariate_{froi}.pdf')
plt.close()



normed_pearsonr
FFA
PPA
EBA

normed_rdmr
FFA
PPA
EBA


## Fig S8: Other category-selective regions

In [45]:
savefig = True

plot_dir = f'{PLOTS_DIR}/section4/supplementary_figures'
os.makedirs(plot_dir, exist_ok=True)

tthreshold = 7
frois = ['OFA', 'OPA', 'FBA', 'VWFA', 'OVWFA']
model_localizers = ['grillspector']

SUBJECTSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/subjectsubject_correlations_nsd515.csv'
MODELSUBJECT_CORRELATIONS_CSV_PATH = f'{RESULTS_DIR}/modelsubject_correlations_nsd515.csv'

all_task_optimized_model_names = ma.get_model_names(model_type='task_optimized')

subjectsubject_correlations_csv = pd.read_csv(SUBJECTSUBJECT_CORRELATIONS_CSV_PATH)
modelsubject_correlations_csv = pd.read_csv(MODELSUBJECT_CORRELATIONS_CSV_PATH)

# frois = subjectsubject_correlations_csv['fmri_froi'].unique()
subjects = subjectsubject_correlations_csv['subject1'].unique()
models = modelsubject_correlations_csv['model_name'].unique()

taskoptimized_models = [model for model in models if model in all_task_optimized_model_names]

hh_color = '#d24c34' # Light peach
# random_color = '#f5e6d3' # Very light brown

localizer_color_dict = {
    'grillspector': '#15a6cc',  # Light blue for Grill-Spector localizer
    'nsd': '#ce9393'            # Light red for NSD localizer
}
localizer_name_dict = {
    'grillspector': 'Category-contrast\n(L1, Stigliani et al., 2015)',
    'kanwisher': 'Kanwisher',
    'nsd': 'Voxel-matched'
}

metrics = ['normed_pearsonr', 'normed_rdmr']
lims_dict = {
    'normed_pearsonr': (0, 1.2),
    'normed_rdmr': (-0.15, 1.75)
}
ticks_dict = {
    'normed_pearsonr': [0, 0.5, 1],
    'normed_rdmr': [0, 0.5, 1, 1.5]
}

# fig, ax = plt.figure(figsize=(5, 5), dpi=300)
fig, axes = plt.subplots(len(metrics), len(frois), figsize=(5*len(frois), 5*len(metrics)), dpi=300)

for midx, metric in enumerate(metrics):
    print()
    print(metric)
    for fidx, froi in enumerate(frois):
        ax = axes[midx, fidx]
        print(froi)
        hh_medians = []
        for subject in subjects:
            subjectsubject_froi = subjectsubject_correlations_csv[
                (subjectsubject_correlations_csv['fmri_froi'] == froi) &
                (subjectsubject_correlations_csv['subject1'] == subject) &
                (subjectsubject_correlations_csv['fmri_tthreshold'] == tthreshold)
            ]
            hh_medians += list(subjectsubject_froi[metric])
        hh_medians = np.array(hh_medians).flatten()

        # Model correlations
        model_corrs = {}
        for model_localizer in model_localizers:
            model_corrs[model_localizer] = {}

            # Separate task-optimized and brain-optimized models
            to_medians = []
            for model in taskoptimized_models:
                modelsubject_froi = modelsubject_correlations_csv[
                    (modelsubject_correlations_csv['fmri_froi'] == froi) &
                    (modelsubject_correlations_csv['model_name'] == model) &
                    (modelsubject_correlations_csv['fmri_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_tthreshold'] == tthreshold) &
                    (modelsubject_correlations_csv['model_localizer'] == model_localizer)
                ]
                model_median = modelsubject_froi[metric].median()
                # Mann-Whitney U test for each model vs. hh_medians
                # ustat, pval = scipy.stats.mannwhitneyu(model_medians, hh_medians, alternative='less')
                # if pval >= 0.05:
                #     print(f"Model {model} ({model_localizer}) is plausibly within the human population (p={pval:.2e})")
                to_medians.append(model_median)
            model_corrs[model_localizer]['task_optimized'] = np.array(to_medians)

        jitter_scale = 0.005 #0.05
        scatter_pad = 0 # 0.05
        alpha = 0.75 # 0.5
        markersize = 15 # 25

        bar_positions = np.array([0, 0.35]) #, 0.7])

        # y=0 line
        ax.axhline(y=0, color='black', linestyle='--', alpha=1, linewidth=0.75, zorder=0)

        # Box plot for brain-brain, and brain-ann for each model localizer
        box_data = [hh_medians]
        for model_localizer in model_localizers:
            box_data.append(model_corrs[model_localizer]['task_optimized'])
        linewidth = 0.75
        barwidth = 0.075 # 0.35
        medianlineextra = 0.075
        boxes = ax.boxplot(
                    box_data, positions=bar_positions, patch_artist=True, 
                    widths=barwidth, showfliers=False,
                    boxprops={'color': 'black', 'linewidth': linewidth},
                    whiskerprops=dict(linewidth=linewidth),
                    capprops=dict(linewidth=linewidth),
                    medianprops={'color': 'black', 'linewidth': linewidth},
                    flierprops=dict(marker='o', markersize=5, linewidth=linewidth)
                )
        for i, data in enumerate(box_data):
            med = np.median(data)
            # Manually draw a longer median line
            x = bar_positions[i]  # X-position of the boxplot
            # 0.225
            ax.plot([x - medianlineextra, x + medianlineextra], [med, med], color='black', linewidth=linewidth, solid_capstyle='butt')
        
        # Color the boxes according to category colors
        colors = [hh_color] + [localizer_color_dict[model_localizer] for model_localizer in model_localizers]
        for box, color in zip(boxes['boxes'], colors):
            box.set(facecolor=color)
        
        # Scatter individual data points
        for i, data in enumerate(box_data):
            x_ = np.zeros(len(data)) + bar_positions[i] + np.random.normal(0, jitter_scale, len(data))
            x_ += scatter_pad
            ax.scatter(
                x_, data, color='gray', alpha=alpha, zorder=5,
                edgecolors='none', s=markersize, linewidths=0.15
            )
            # print(f'Bar position {bar_positions[i]}: n={len(data)}')
        
        # Write froi name in bottom left corner
        ax.text(0.05, 0.25, froi, transform=ax.transAxes, fontsize=12, verticalalignment='top')

        yticks = np.array(ticks_dict[metric])
        ylims = lims_dict[metric]
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_position(('outward', 5))
        ax.spines['bottom'].set_position(('outward', 5))
        ax.set_yticks(yticks)
        ax.set_yticklabels(yticks, fontsize=12)
        ax.spines['left'].set_bounds(yticks.min(), yticks.max())
        # plt.tick_params(axis='y', labelleft=False)
        ax.set_ylim(ylims)
        ax.spines['bottom'].set_bounds(bar_positions[0], bar_positions[-1])
        ax.set_xlim(bar_positions[0]-0.15, bar_positions[-1]+0.15)
        # ax.tick_params(axis='x', labelbottom=False)

        if fidx != 0:
            ax.spines['left'].set_visible(False)
            # Stop showing y-axis ticks
            ax.yaxis.set_ticks([])
        else:
            if metric == 'normed_pearsonr':
                ax.set_ylabel(r'Noise-corrected pearson $\rho$', fontsize=14)
            else:
                ax.set_ylabel(r'Noise-corrected RDM $\rho$', fontsize=14)
        # x-label on all plots for now
        # ax.set_xticks(bar_positions, ['Brain-Brain', 'Brain-ANN\n(Task-Optimized)', 'Brain-ANN\n(Brain-Optimized)'], fontsize=12)
        xtick_labels = ['Brain-Brain'] + \
            [f'Brain-ANN' for model_localizer in model_localizers]
        ax.set_xticks(bar_positions)
        ax.set_xticklabels(xtick_labels, fontsize=10, rotation=0, ha='center')

plt.tight_layout()
if savefig:
    # raise NotImplementedError("Don't overwrite existing fig3_sup_otherfrois.pdf without checking it first!")
    plt.savefig(f'{plot_dir}/figS8_otherfrois.pdf', dpi=300, bbox_inches='tight', transparent=True)
else:
    plt.show()
# plt.savefig(f'{plot_dir}/modelhuman_univariate_{froi}.pdf')
plt.close()



normed_pearsonr
OFA
OPA
FBA
VWFA
OVWFA

normed_rdmr
OFA
OPA
FBA
VWFA
OVWFA
